# Walking a HuBMAP dataset through Globus — without downloading it

**What this does.** Globus Transfer has an `operation_ls` call that lists one directory:
name, type, size, last-modified. It moves **no file content**. Walk the tree with it and you
get a complete file manifest for a dataset you have never downloaded.

**Why it matters.** HuBMAP publishes a file manifest for HIVE-processed datasets only
(2,630 of them). For the 8,362 raw/uploaded datasets there is none. This is how you make one.

**The one thing to understand:** `operation_ls` is **not recursive**. One call = one directory.
To get the whole tree you call it once per directory and manage the queue yourself. That is
all `walk()` below does.

---

## How to work through this notebook

| section | what you do |
|---|---|
| 1 | install + imports |
| 2 | log in to Globus (once per session) |
| 3 | find the HuBMAP collection UUID (once, ever — write it down) |
| 4 | resolve a dataset's path from its HuBMAP ID |
| 5 | **list ONE directory** — do this first, to see the raw shape of a response |
| 6 | walk the whole tree |
| 7 | explore what you got |
| 8 | compare against HuBMAP's own manifest (processed datasets only) |
| 9 | save it |

Run 5 before 6. The single-directory call is where you learn what the data looks like;
the walk is just that call in a loop.


## 1. Setup


In [ ]:
# !pip install globus-sdk

import json, urllib.request, collections, os
from pathlib import Path

import globus_sdk
from globus_sdk.scopes import TransferScopes

print('globus-sdk', globus_sdk.__version__)


## 2. Log in

A *native app* login: the cell prints a URL, you approve in the browser, paste the code back.
The token lives in memory for this kernel only — rerun this cell after a kernel restart.

`CLIENT_ID` below is Globus's public tutorial client, which is fine for read-only listing.
If your institution blocks it, register your own at
[app.globus.org/settings/developers](https://app.globus.org/settings/developers) and swap the value.


In [ ]:
CLIENT_ID = '61338d24-54d5-408f-a10d-66c06b59f6d2'

auth = globus_sdk.NativeAppAuthClient(CLIENT_ID)
auth.oauth2_start_flow(requested_scopes=TransferScopes.all)

print('Open this URL, log in, then paste the code below:')
print()
print(auth.oauth2_get_authorize_url())


In [ ]:
code_str = input('code: ').strip()

tokens = auth.oauth2_exchange_code_for_tokens(code_str)
tk = tokens.by_resource_server['transfer.api.globus.org']
tc = globus_sdk.TransferClient(
    authorizer=globus_sdk.AccessTokenAuthorizer(tk['access_token'])
)
print('logged in')


## 3. Find the HuBMAP collection UUID

This is **not** exposed by any HuBMAP API — I checked the entity API and the portal JSON.
Two ways to get it:

1. Search Globus (below). Results can be ambiguous, so read the display names.
2. Open any dataset's portal page, click through to Globus, and read `origin_id=` out of the URL.

Once you have it, paste it into `COLLECTION` and never do this again.


In [ ]:
for ep in tc.endpoint_search('hubmap', filter_scope='all', limit=25):
    print(f"{ep['id']}  {ep['display_name']}")
    print(f"     owner: {ep.get('owner_string')}   type: {ep.get('entity_type')}")


In [ ]:
COLLECTION = 'PASTE-THE-UUID-HERE'


## 4. Resolve a dataset's path

HuBMAP's entity API turns a human ID (`HBM279.TQRS.775`) into the path inside the collection.
The field is `local_directory_rel_path`, e.g.

```
public/University of Florida TMC/077f7862f6306055899374c7807a30c3/
```

Note the **spaces** in the group name. Pass the path to the SDK as a plain string — it handles
the encoding. Do not hand-build a URL.


In [ ]:
DATASET = 'HBM279.TQRS.775'      # raw CODEX, 13k files  — no HuBMAP manifest exists
# DATASET = 'HBM695.NCKX.893'    # processed, 96 files   — has a manifest to compare against
# DATASET = 'HBM653.RRCF.859'    # processed, 651 files

def hubmap_entity(identifier):
    url = 'https://entity.api.hubmapconsortium.org/entities/' + identifier
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.load(r)

ent  = hubmap_entity(DATASET)
ROOT = '/' + ent['local_directory_rel_path'].strip('/')

print(ent['hubmap_id'], '|', ent['uuid'])
print('access :', ent.get('data_access_level'), '|', ent.get('status'))
print('path   :', ROOT)


## 5. List ONE directory first

Do this before the walk. It shows you exactly what a response contains, and it confirms your
collection UUID and path are right — a 30-second check instead of a failed 40-minute walk.

Each entry is a dict. The fields you care about:

| field | meaning |
|---|---|
| `name` | basename only, not a path |
| `type` | `'file'`, `'dir'`, or `'invalid_symlink'` |
| `size` | bytes (0 for directories) |
| `last_modified` | string timestamp |

`show_hidden=True` is deliberate: dotfiles like `.DS_Store` are real drift and you want to see them.


In [ ]:
entries = list(tc.operation_ls(COLLECTION, path=ROOT, show_hidden=True))

print(f'{len(entries)} entries in {ROOT}')
print()
for e in entries:
    kind = 'DIR ' if e['type'] == 'dir' else 'file'
    size = '' if e['type'] == 'dir' else f"{e['size']:>14,} B"
    print(f"  {kind}  {e['name']:<50} {size}")


Look at one raw entry to see everything the API returns:


In [ ]:
print(json.dumps(entries[0], indent=2))


## 6. Walk the whole tree

Depth-first. A stack of directories to visit; pop one, list it, push any subdirectories, record
any files. That is the entire algorithm — `operation_ls` gives you no recursion, so this is it.

**Cost.** One call per directory. Roughly 30–50 calls for a raw CODEX dataset, ~6 for a processed
one. About 4,000 calls to cover all 128 raw CODEX datasets — an hour, not a project.

Paths in the result are **relative to `root`**, which is what you want: the person who downloads
this will have it under a directory name of their own choosing.


In [ ]:
def walk(tc, collection, root, verbose=True):
    """Every file under `root`, as rel_path + size + mtime. Transfers no data."""
    root  = root.rstrip('/')
    files, errors = [], []
    queue = [root]
    ndirs = 0

    while queue:
        d = queue.pop()
        ndirs += 1
        try:
            entries = tc.operation_ls(collection, path=d, show_hidden=True)
        except globus_sdk.TransferAPIError as e:
            errors.append((d, e.code, e.message))
            continue

        for e in entries:
            full = f"{d}/{e['name']}"
            if e['type'] == 'dir':
                queue.append(full)
            else:
                files.append({
                    'rel_path':      full[len(root) + 1:],
                    'size':          e.get('size'),
                    'type':          e.get('type'),
                    'last_modified': e.get('last_modified'),
                })
        if verbose:
            print(f'\r  dirs {ndirs}  files {len(files)}', end='')

    if verbose:
        print()
    return files, errors


In [ ]:
files, errors = walk(tc, COLLECTION, ROOT)

total = sum(f['size'] or 0 for f in files)
print(f'\n{len(files):,} files   {total:,} bytes   ({total/1e9:.2f} GB)')
if errors:
    print(f'{len(errors)} directories could not be listed:')
    for d, c, m in errors[:5]:
        print('   ', d, '->', c, m)


> **Publish the byte count, not a rounded figure.** 66,436,899,861 bytes is 66.44 GB *or*
> 61.9 GiB depending on who is counting. Rounded numbers manufacture false mismatches.


## 7. Explore what you got

Four questions worth asking of any tree before you write a single FileSet.


**7a. What formats are in here, and how much of each?**


In [ ]:
by_ext = collections.Counter(os.path.splitext(f['rel_path'])[1].lower() or '(none)' for f in files)
bytes_by_ext = collections.Counter()
for f in files:
    bytes_by_ext[os.path.splitext(f['rel_path'])[1].lower() or '(none)'] += f['size'] or 0

print(f"{'ext':<12}{'files':>8}{'GB':>10}")
for ext, n in by_ext.most_common():
    print(f'{ext:<12}{n:>8,}{bytes_by_ext[ext]/1e9:>10.2f}')


**7b. What is the top-level shape?**


In [ ]:
by_top = collections.Counter(f['rel_path'].split('/')[0] for f in files)
for top, n in by_top.most_common():
    print(f'{n:>7,}  {top}')


**7c. The structural signature.**

The set of `(directory, extension)` pairs. This is how you tell whether two datasets share a
layout. Across 172 processed CODEX datasets there are only **three** distinct signatures — so
a convention written once covers dozens of datasets. Key your conventions on this, never on
`dataset_type`: layouts drift between pipeline versions under one label.


In [ ]:
def signature(files, depth=1):
    out = set()
    for f in files:
        parts = f['rel_path'].split('/')
        d = '/'.join(parts[:depth]) if len(parts) > depth else '(root)'
        out.add((d, os.path.splitext(parts[-1])[1].lower() or '(none)'))
    return frozenset(out)

sig = signature(files)
print(f'{len(sig)} (dir, ext) pairs')
for d, e in sorted(sig):
    print(f'  {d:<40} {e}')


**7d. Where is the weight, and what is odd?**

Heterogeneity warnings — a directory holding several extensions is a hint that it is *not* one
FileSet. HBM279's `diagnostics/` folder holds 21 `.txt`, 1 `.log` and 1,712 `.tif` across four
unrelated purposes; declaring it as one FileSet with `encodingFormat: image/tiff` is simply false.


In [ ]:
print('--- 10 biggest files ---')
for f in sorted(files, key=lambda x: -(x['size'] or 0))[:10]:
    print(f"{(f['size'] or 0)/1e6:>10.1f} MB  {f['rel_path']}")

print()
print('--- directories holding more than one extension ---')
mixed = collections.defaultdict(set)
for f in files:
    mixed[os.path.dirname(f['rel_path'])].add(os.path.splitext(f['rel_path'])[1].lower())
for d, exts in sorted(mixed.items()):
    if len(exts) > 1:
        print(f'  {d or "(root)":<55} {sorted(exts)}')


In [ ]:
print('--- names differing only by case (a trap on macOS / Windows) ---')
base = collections.defaultdict(set)
for f in files:
    base[os.path.basename(f['rel_path']).lower()].add(os.path.basename(f['rel_path']))
for k, v in base.items():
    if len(v) > 1:
        print('  ', sorted(v))

print()
print('--- paths with spaces, and the longest path ---')
for f in files:
    if ' ' in f['rel_path']:
        print('  ', repr(f['rel_path']))
longest = max(files, key=lambda f: len(f['rel_path']))
print(f"  longest: {len(longest['rel_path'])} chars (Windows limit is 260)")


## 8. Compare against HuBMAP's own manifest

Processed datasets carry a `files` array in the search index. Raw ones do not. This cell answers
the open question: **does that manifest exactly describe what Globus serves?**

Run it on `HBM695.NCKX.893` — set `DATASET` in section 4 and rerun from there.

Expect some benign noise: hidden files the indexer may filter, and index lag (the record carries
its own `index_version` and can trail the filesystem).


In [ ]:
def hubmap_api_manifest(uuid):
    payload = {'query': {'ids': {'values': [uuid]}}, '_source': ['files'], 'size': 1}
    req = urllib.request.Request(
        'https://search.api.hubmapconsortium.org/v3/portal/search',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=60) as r:
        hits = json.load(r)['hits']['hits']
    return hits[0]['_source'].get('files') if hits else None

api = hubmap_api_manifest(ent['uuid'])

if api is None:
    print('No `files` manifest in the index — expected for a raw dataset.')
else:
    g = {f['rel_path']: f.get('size') for f in files}
    a = {f['rel_path']: f.get('size') for f in api}
    only_g, only_a = sorted(set(g) - set(a)), sorted(set(a) - set(g))
    mismatch = [(p, g[p], a[p]) for p in set(g) & set(a) if g[p] != a[p]]

    print(f'Globus {len(g)} files   HuBMAP API {len(a)} files')
    print(f'  only in Globus  {len(only_g)}')
    print(f'  only in API     {len(only_a)}')
    print(f'  size mismatches {len(mismatch)}')
    for label, rows in (('ONLY IN GLOBUS', only_g), ('ONLY IN API', only_a)):
        for p in rows[:10]:
            print(f'    [{label}] {p}')
    for p, gs, as_ in mismatch[:10]:
        print(f'    [SIZE] {p}  globus={gs} api={as_}')
    if not (only_g or only_a or mismatch):
        print('\n  IDENTICAL — the API manifest exactly describes what Globus serves.')


## 9. Save it

This JSON is the artifact. It is what HuBMAP does not publish for raw datasets, and it is what
a Croissant `FileSet` inventory should be built from.


In [ ]:
out = Path(f"{ent['hubmap_id']}.globus-manifest.json")
out.write_text(json.dumps({
    'hubmap_id': ent['hubmap_id'],
    'uuid':      ent['uuid'],
    'root':      ROOT,
    'n_files':   len(files),
    'n_bytes':   sum(f['size'] or 0 for f in files),
    'files':     sorted(files, key=lambda f: f['rel_path']),
}, indent=2))

print(f'[write] {out}  ({out.stat().st_size/1e6:.1f} MB)')


---

## Troubleshooting

| symptom | cause |
|---|---|
| `ConsentRequired` on `operation_ls` | The collection needs a data-access consent. Rerun section 2 — the error carries the extra scope to request. |
| `EndpointNotFound` / `ClientError.NotFound` | Wrong `COLLECTION` UUID, or the path does not exist. Rerun section 5 on `ROOT` alone. |
| `PermissionDenied` | The dataset is consortium or protected, not public. Check `data_access_level` from section 4. |
| The walk hangs | Some collections rate-limit. Add a short `time.sleep()` in the loop. |
| Zero files, no error | `ROOT` pointed at a file, or at an empty directory. Print `entries` from section 5. |

## What this does *not* give you

`operation_ls` returns **no checksum** — I checked the SDK. You get path, size and mtime.
Globus verifies checksums *during a transfer* (`verify_checksum=True`, `sync_level='checksum'`),
which proves the copy matches the source, but it never publishes a digest you can put in metadata.

For digests you need either direct filesystem access to `/hive/hubmap/data/public/<uuid>`
(try `GET https://ingest.api.hubmapconsortium.org/datasets/<uuid>/file-system-abs-path`),
or you compute them once from your own complete download.
